In [1]:
import pandas as pd
import numpy as np

train_path = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_train_cleaned.csv"
test_path  = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_test_cleaned.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (125973, 44)
Test shape: (22544, 44)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,binary_label
0,-0.110249,-0.124706,-0.686785,0.751111,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387,normal,20,normal
1,-0.110249,2.219312,0.781428,0.751111,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,normal,15,normal
2,-0.110249,-0.124706,1.087305,-0.736235,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.174417,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387,neptune,19,attack
3,-0.110249,-0.124706,-0.442083,0.751111,-0.007723,-0.002891,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.439078,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084,normal,21,normal
4,-0.110249,-0.124706,-0.442083,0.751111,-0.007728,-0.004814,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.439078,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,normal,21,normal


In [2]:
# Separate features (X) from the target label (y)
feature_cols = [col for col in train_df.columns if col not in ['label', 'binary_label', 'difficulty']]

X_train = train_df[feature_cols]
y_train = train_df['binary_label']

X_test = test_df[feature_cols]
y_test = test_df['binary_label']

print("X_train shape:", X_train.shape)
print("y_train distribution:")
print(y_train.value_counts())

X_train shape: (125973, 41)
y_train distribution:
binary_label
normal    67343
attack    58630
Name: count, dtype: int64


In [3]:
print(train_df.columns.tolist())

['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty', 'binary_label']


In [4]:
# Recreate the binary label from the original label column
train_df['binary_label'] = train_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['binary_label']  = test_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print(train_df['binary_label'].value_counts())

binary_label
normal    67343
attack    58630
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix

# Define the five algorithms, each with a fixed random_state for reproducibility
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Stratified 5-fold cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Models and cross-validation setup ready!")
print("Models:", list(models.keys()))

Models and cross-validation setup ready!
Models: ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'SVM']


In [6]:
import time

# Store fold-level results for every model and metric
results = {name: {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'fpr': [], 'mcc': []} for name in models}

X_train_array = X_train.values
y_train_array = y_train.values

for fold_num, (train_idx, val_idx) in enumerate(skf.split(X_train_array, y_train_array), 1):
    X_fold_train, X_fold_val = X_train_array[train_idx], X_train_array[val_idx]
    y_fold_train, y_fold_val = y_train_array[train_idx], y_train_array[val_idx]
    
    print(f"\n--- Fold {fold_num}/5 ---")
    
    for name, model in models.items():
        start = time.time()
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        elapsed = time.time() - start
        
        acc = accuracy_score(y_fold_val, y_pred)
        prec = precision_score(y_fold_val, y_pred, pos_label='attack')
        rec = recall_score(y_fold_val, y_pred, pos_label='attack')
        f1 = f1_score(y_fold_val, y_pred, pos_label='attack')
        mcc = matthews_corrcoef(y_fold_val, y_pred)
        
        tn, fp, fn, tp = confusion_matrix(y_fold_val, y_pred, labels=['normal', 'attack']).ravel()
        fpr = fp / (fp + tn)
        
        results[name]['accuracy'].append(acc)
        results[name]['precision'].append(prec)
        results[name]['recall'].append(rec)
        results[name]['f1'].append(f1)
        results[name]['fpr'].append(fpr)
        results[name]['mcc'].append(mcc)
        
        print(f"{name}: acc={acc:.4f}, f1={f1:.4f}, mcc={mcc:.4f} ({elapsed:.1f}s)")

print("\nAll folds complete!")


--- Fold 1/5 ---
Logistic Regression: acc=0.9568, f1=0.9532, mcc=0.9133 (1.5s)
KNN: acc=0.9956, f1=0.9953, mcc=0.9912 (6.5s)
Decision Tree: acc=0.9982, f1=0.9981, mcc=0.9964 (1.0s)
Random Forest: acc=0.9992, f1=0.9992, mcc=0.9985 (10.9s)
SVM: acc=0.9917, f1=0.9911, mcc=0.9834 (60.1s)

--- Fold 2/5 ---
Logistic Regression: acc=0.9517, f1=0.9474, mcc=0.9031 (1.4s)
KNN: acc=0.9962, f1=0.9959, mcc=0.9923 (4.6s)
Decision Tree: acc=0.9979, f1=0.9977, mcc=0.9958 (1.2s)
Random Forest: acc=0.9989, f1=0.9988, mcc=0.9978 (10.0s)
SVM: acc=0.9919, f1=0.9913, mcc=0.9838 (53.2s)

--- Fold 3/5 ---
Logistic Regression: acc=0.9523, f1=0.9482, mcc=0.9043 (1.4s)
KNN: acc=0.9957, f1=0.9954, mcc=0.9914 (4.6s)
Decision Tree: acc=0.9976, f1=0.9974, mcc=0.9952 (1.1s)
Random Forest: acc=0.9986, f1=0.9985, mcc=0.9971 (10.0s)
SVM: acc=0.9910, f1=0.9903, mcc=0.9819 (51.2s)

--- Fold 4/5 ---
Logistic Regression: acc=0.9520, f1=0.9479, mcc=0.9036 (1.6s)
KNN: acc=0.9956, f1=0.9953, mcc=0.9911 (4.7s)
Decision Tree: a

In [7]:
import pandas as pd

# Average each metric across the 5 folds, per model
summary = pd.DataFrame({
    name: {metric: sum(vals)/len(vals) for metric, vals in results[name].items()}
    for name in results
}).T

summary = summary.round(4)
print(summary)

                     accuracy  precision  recall      f1     fpr     mcc
Logistic Regression    0.9539     0.9599  0.9402  0.9499  0.0342  0.9073
KNN                    0.9958     0.9958  0.9953  0.9955  0.0037  0.9916
Decision Tree          0.9980     0.9979  0.9977  0.9978  0.0018  0.9960
Random Forest          0.9989     0.9994  0.9982  0.9988  0.0005  0.9978
SVM                    0.9916     0.9922  0.9898  0.9910  0.0068  0.9832


In [8]:
from scipy.stats import wilcoxon
from itertools import combinations

model_names = list(results.keys())
print("Wilcoxon signed-rank test results (F1 score, pairwise):\n")

for model_a, model_b in combinations(model_names, 2):
    scores_a = results[model_a]['f1']
    scores_b = results[model_b]['f1']
    
    stat, p_value = wilcoxon(scores_a, scores_b)
    
    significant = "SIGNIFICANT (p < 0.05)" if p_value < 0.05 else "not significant"
    print(f"{model_a} vs {model_b}: p={p_value:.4f} -> {significant}")

Wilcoxon signed-rank test results (F1 score, pairwise):

Logistic Regression vs KNN: p=0.0625 -> not significant
Logistic Regression vs Decision Tree: p=0.0625 -> not significant
Logistic Regression vs Random Forest: p=0.0625 -> not significant
Logistic Regression vs SVM: p=0.0625 -> not significant
KNN vs Decision Tree: p=0.0625 -> not significant
KNN vs Random Forest: p=0.0625 -> not significant
KNN vs SVM: p=0.0625 -> not significant
Decision Tree vs Random Forest: p=0.0625 -> not significant
Decision Tree vs SVM: p=0.0625 -> not significant
Random Forest vs SVM: p=0.0625 -> not significant


In [9]:
# Stratified 10-fold cross-validation setup
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("Cross-validation updated to 10 folds")

Cross-validation updated to 10 folds


In [10]:
# Save the fold-level results and summary table
import os

results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\results"
os.makedirs(results_folder, exist_ok=True)

# Save summary table-
summary.to_csv(os.path.join(results_folder, "Phase1_NSLKDD_summary.csv"))

# Save fold-level results too (for the Wilcoxon test / future reference)
fold_results = pd.DataFrame({
    (name, metric): vals
    for name in results
    for metric, vals in results[name].items()
})
fold_results.to_csv(os.path.join(results_folder, "Phase1_NSLKDD_foldlevel.csv"))

print("Results saved successfully!")
print("Summary:", os.path.join(results_folder, "Phase1_NSLKDD_summary.csv"))
print("Fold-level:", os.path.join(results_folder, "Phase1_NSLKDD_foldlevel.csv"))

Results saved successfully!
Summary: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\results\Phase1_NSLKDD_summary.csv
Fold-level: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\results\Phase1_NSLKDD_foldlevel.csv
